In [ ]:
import gc
import math
import os
import pickle
import platform
import random
import re
import time
import numpy as np
import pandas as pd
import torch
import pickle
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
import hashlib
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import lightgbm as lgb
from catboost import CatBoostRanker, Pool

DATA = '.'                      
SEED = 42   
N_FOLDS = 5    
print('System:', platform.platform())
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0),
      '| память: %.0f ГБ' % (torch.cuda.get_device_properties(0).total_memory/1e9))
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

System: Windows-10-10.0.19045-SP0
torch 2.11.0+cu128 | CUDA 12.8 | GPU: NVIDIA GeForce RTX 5090 | память: 34 ГБ


# в одной папке должны лежать train.csv, test.csv и sample_submission.csv

## Данные и признаки 

In [ ]:
CACHE='feat_cache.pkl'
if os.path.exists(CACHE):
    with open(CACHE,'rb') as f: alldf,F,sub=pickle.load(f)
    print('признаки загружены из кэша:',F.shape)
else:
    train=pd.read_csv(f'{DATA}/train.csv'); test=pd.read_csv(f'{DATA}/test.csv'); sub=pd.read_csv(f'{DATA}/sample_submission.csv')
    print(train.shape, test.shape)
    print(train.isna().mean()[lambda s:s>0])
    print(train.groupby('ingest_source').relevance.agg(['mean','size']))
    print(train.groupby('query_id').size().describe())

    JUNK=[r'Главная\s*›\s*Энциклопедия', r'\[\d+\]', r'\s*\n+\s*']
    def clean(s):
        if not isinstance(s,str): return ''
        s=s.strip().strip('"').replace('\xa0',' ')
        s=re.sub(r'Главная\s*›\s*Энциклопедия','',s)
        s=re.sub(r'\[\d+\]','',s)
        s=re.sub(r'<[^>]+>',' ',s)
        s=re.sub(r'\s+',' ',s)
        return s.strip()
    def prep(df):
        df=df.copy()
        for c in ['query','title','text']: df[c+'_c']=df[c].map(clean)
        for c in ['passage_index','article_passage_count','article_char_count']:
            df[c]=pd.to_numeric(df[c],errors='coerce')
        df['ingest_source']=df['ingest_source'].fillna('NA').astype(str)
        return df
    train=prep(train); test=prep(test)

    
    STOP=set('и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между ли какой какие сколько'.split())
    def toks(s, k=5):
        w=re.findall(r'[а-яёa-z0-9]+', s.lower().replace('ё','е'))
        return [x[:k] for x in w if x not in STOP]

    class BM25:
        def __init__(s, docs, k1=1.5, b=0.75):
            s.k1,s.b=k1,b; s.docs=[Counter(d) for d in docs]; s.len=np.array([len(d) for d in docs])+1e-9
            s.avg=s.len.mean(); df=Counter(); [df.update(d.keys()) for d in s.docs]
            N=len(docs); s.idf={t:math.log(1+(N-n+.5)/(n+.5)) for t,n in df.items()}
        def score(s,q,i):
            d=s.docs[i]; r=0
            for t in q:
                f=d.get(t,0)
                if f: r+=s.idf[t]*f*(s.k1+1)/(f+s.k1*(1-s.b+s.b*s.len[i]/s.avg))
            return r

    def lex_feats(df):
        
        dd=df.drop_duplicates('document_id')
        dtok={d:toks(t) for d,t in zip(dd.document_id,dd.text_c)}
        ids=list(dtok); pos={d:i for i,d in enumerate(ids)}
        bm_t=BM25([dtok[d] for d in ids])
        ar=df.drop_duplicates('article_id'); atok={a:toks(t) for a,t in zip(ar.article_id,ar.title_c)}
        aid=list(atok); apos={a:i for i,a in enumerate(aid)}
        bm_h=BM25([atok[a] for a in aid])
        rows=[]
        for q,t,d,a,tt in zip(df.query_c,df.title_c,df.document_id,df.article_id,df.text_c):
            qt=toks(q); st=set(qt); dt=set(dtok[d]); ht=set(atok[a])
            ov=len(st&dt); 
            rows.append(dict(bm_text=bm_t.score(qt,pos[d]), bm_title=bm_h.score(qt,apos[a]),
                ov_text=ov, ov_text_r=ov/max(len(st),1), ov_title=len(st&ht), ov_title_r=len(st&ht)/max(len(st),1),
                q_len=len(qt), t_len=len(dtok[d]), text_chars=len(tt), title_in_query=float(t.lower() in q.lower()) if t else 0.,
                q_has_digit=float(bool(re.search(r'\d',q))), text_n_digits=len(re.findall(r'\d+',tt)),
                text_missing=float(len(tt)==0)))
        return pd.DataFrame(rows,index=df.index)

    
    def tfidf_feats(df, fit_texts):
        out=pd.DataFrame(index=df.index)
        for name,(an,ng) in {'w':('word',(1,2)),'c':('char_wb',(3,5))}.items():
            v=TfidfVectorizer(analyzer=an,ngram_range=ng,sublinear_tf=True,min_df=2,lowercase=True,max_features=300000).fit(fit_texts)
            Q=v.transform(df.query_c); T=v.transform(df.text_c); H=v.transform(df.title_c)
            out[f'tf_{name}_text']=np.asarray(Q.multiply(T).sum(1)).ravel()
            out[f'tf_{name}_title']=np.asarray(Q.multiply(H).sum(1)).ravel()
        return out

    alldf=pd.concat([train.assign(_s='train'),test.assign(_s='test',relevance=np.nan)],ignore_index=True)
    F=lex_feats(alldf)
    F=F.join(tfidf_feats(alldf, pd.concat([alldf.text_c,alldf.title_c,alldf.query_c])))
    meta=alldf[['passage_index','article_passage_count','article_char_count']].copy()
    meta['pos_rel']=meta.passage_index/meta.article_passage_count.clip(lower=1)
    meta['src']=alldf.ingest_source.astype('category').cat.codes
    F=F.join(meta)
    
    g=alldf.query_id
    for c in ['bm_text','bm_title','tf_w_text','tf_c_text','tf_w_title','ov_text_r']:
        F[c+'_rank']=F.groupby(g)[c].rank(ascending=False,method='min')
        F[c+'_z']=(F[c]-F.groupby(g)[c].transform('mean'))/(F.groupby(g)[c].transform('std').fillna(0)+1e-6)
        F[c+'_dmax']=F[c]-F.groupby(g)[c].transform('max')
    F['n_cand']=g.map(g.value_counts())
    F['n_same_article']=alldf.groupby(['query_id','article_id']).document_id.transform('count')
    F['doc_freq_queries']=alldf.document_id.map(alldf.document_id.value_counts())
    F['label_noise_flag']=0
    F=F.drop(columns=['label_noise_flag'])
    print(F.shape)
    with open(CACHE,'wb') as f: pickle.dump((alldf,F,sub),f)
    print('признаки посчитаны и сохранены в кэш:',F.shape)

(31530, 12) (13507, 11)
title                    0.044561
passage_index            0.044339
article_passage_count    0.062131
article_char_count       0.080368
dtype: float64
                   mean   size
ingest_source                 
S0             0.258997  15977
S1             0.261365   9481
S2             0.260779   3827
S3             0.246771   2245
count    3042.000000
mean       10.364892
std         1.072353
min         2.000000
25%        10.000000
50%        10.000000
75%        11.000000
max        19.000000
dtype: float64
(45037, 43)
признаки посчитаны и сохранены в кэш: (45037, 43)


## Метрика и фиксированные фолды

In [ ]:
def ndcg10(df, pred):
    d = pd.DataFrame({'q': df.query_id.values, 'id': df.id.values, 'y': df.relevance.values, 'p': pred})
    res = []
    for q, g in d.groupby('q'):
        g = g.sort_values(['p', 'id'], ascending=[False, True])
        gain = 2.0 ** g.y.values - 1; disc = 1 / np.log2(np.arange(2, len(g) + 2))
        idcg = (np.sort(gain)[::-1][:10] * disc[:10]).sum()
        res.append((gain[:10] * disc[:10]).sum() / idcg if idcg > 0 else 0.)
    return float(np.mean(res))


tr_idx = np.where(alldf._s == 'train')[0]; te_idx = np.where(alldf._s == 'test')[0]


_fold_of = np.array([int(hashlib.md5(x.encode()).hexdigest(), 16) % N_FOLDS for x in alldf.query_id.values[tr_idx]])
folds = [(np.where(_fold_of != k)[0], np.where(_fold_of == k)[0]) for k in range(N_FOLDS)]
print('размеры валидационных фолдов (строк):', [len(v) for _, v in folds])

размеры валидационных фолдов (строк): [6228, 5895, 6353, 6312, 6742]


## bge-reranker 

In [ ]:

a_all=alldf.query_c.tolist(); b_all=(alldf.title_c+'. '+alldf.text_c).tolist()
y_all=alldf.relevance.values; q_all=alldf.query_id.values

def run_ce(tag, model_name, maxlen, epochs, lr, qpb, accum, pred_bs=64, save_every=200, seed_off=0, grad_ckpt=True):
    final=f'{tag}_final.npy'
    if os.path.exists(final):
        print(f'[{tag}] уже посчитано, загружаю {final}'); return np.load(final)
    tok=AutoTokenizer.from_pretrained(model_name)
    def enc(idx):
        return tok([a_all[i] for i in idx],[b_all[i] for i in idx],truncation='only_second',max_length=maxlen,padding=True,return_tensors='pt')
    @torch.no_grad()
    def predict(model, idx):
        model.eval(); idx=np.asarray(idx); order=np.argsort([len(b_all[i]) for i in idx]); out=np.zeros(len(idx),dtype=np.float32)
        for s in range(0,len(idx),pred_bs):
            o=order[s:s+pred_bs]; x={k:v.to('cuda') for k,v in enc(idx[o]).items()}
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                out[o]=model(**x).logits.squeeze(-1).float().cpu().numpy()
        return out
    ckpt=f'{tag}_ckpt.pt'
    def train_fold(train_idx, fold):
        torch.manual_seed(SEED+fold+seed_off)
        model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=1).to('cuda')
        if grad_ckpt: model.gradient_checkpointing_enable()
        opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
        groups={}
        for i in train_idx: groups.setdefault(q_all[i],[]).append(i)
        qs0=sorted(groups); n_b=math.ceil(len(qs0)/qpb); total_b=epochs*n_b; n_steps=math.ceil(total_b/accum)
        sch=get_linear_schedule_with_warmup(opt,int(0.06*n_steps),n_steps)
        scaler=torch.amp.GradScaler('cuda'); bce=torch.nn.BCEWithLogitsLoss()
        start=0
        if os.path.exists(ckpt):
            ck=torch.load(ckpt,map_location='cpu')
            if ck['fold']==fold:
                model.load_state_dict(ck['model']); start=ck['batch']
                for _ in range(start//accum): sch.step()
                print(f'>>> [{tag}] продолжаем фолд {fold+1} с батча {start}/{total_b}')
            del ck; gc.collect()
        gb=0; tot=0; nbl=0; t0=time.time()
        for ep in range(epochs):
            qs=list(qs0); np.random.RandomState(SEED*100+fold*10+ep+seed_off).shuffle(qs); model.train()
            for bi in range(0,len(qs),qpb):
                if gb<start: gb+=1; continue
                bq=qs[bi:bi+qpb]; idx=np.concatenate([groups[q] for q in bq])
                x={k:v.to('cuda') for k,v in enc(idx).items()}; y=torch.tensor(y_all[idx],dtype=torch.float32,device='cuda')
                with torch.autocast(device_type='cuda',dtype=torch.float16):
                    logits=model(**x).logits.squeeze(-1)
                logits=logits.float(); loss=0; p=0
                for q in bq:
                    n=len(groups[q]); l=logits[p:p+n]; yy=y[p:p+n]; p+=n
                    if yy.sum()>0: loss=loss-(l[yy==1]-torch.logsumexp(l,0)).mean()
                loss=loss/len(bq)+0.3*bce(logits,y)
                scaler.scale(loss/accum).backward(); tot+=loss.item(); nbl+=1; gb+=1
                if gb%accum==0:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
                if gb%100==0: print(f'  [{tag}] батч {gb}/{total_b} loss {tot/max(nbl,1):.4f} {time.time()-t0:.0f}s',flush=True)
                if gb%save_every==0:
                    torch.save({'fold':fold,'batch':gb,'model':model.state_dict()},ckpt+'.tmp'); os.replace(ckpt+'.tmp',ckpt)
        return model
    ce=np.load(f'{tag}_partial.npy') if os.path.exists(f'{tag}_partial.npy') else np.zeros(len(alldf))
    done=int(np.load(f'{tag}_done.npy')) if os.path.exists(f'{tag}_done.npy') else 0
    for k,(tri,vai) in enumerate(folds):
        if k<done: continue
        print(f'=== [{tag}] фолд {k+1}/{N_FOLDS} ===')
        ti,vi=tr_idx[tri],tr_idx[vai]; model=train_fold(ti,k)
        ce[vi]=predict(model,vi); ce[te_idx]+=predict(model,te_idx)/N_FOLDS
        print(f'[{tag}] fold NDCG@10:',ndcg10(alldf.iloc[vi],ce[vi]))
        np.save(f'{tag}_partial.npy',ce); np.save(f'{tag}_done.npy',k+1)
        if os.path.exists(ckpt): os.remove(ckpt)
        del model; gc.collect(); torch.cuda.empty_cache()
    np.save(final,ce); return ce

c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ce_bge = run_ce('bge5f_e2','BAAI/bge-reranker-v2-m3',maxlen=384,epochs=2,lr=2e-5,qpb=2,accum=4,pred_bs=64,seed_off=0,grad_ckpt=True)
gc.collect(); torch.cuda.empty_cache()
print('bge OOF NDCG:',ndcg10(alldf.iloc[tr_idx],ce_bge[tr_idx]))


=== [bge5f_e2] фолд 1/5 ===


W1114 01:45:22.360000 9688 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 626.14it/s]
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
C:\Users\Mikhail\AppData\Local\Temp\ipykernel_9688\81505692.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytor

  [bge5f_e2] батч 100/2438 loss 1.9559 31s
  [bge5f_e2] батч 200/2438 loss 1.8023 53s
  [bge5f_e2] батч 300/2438 loss 1.7009 75s
  [bge5f_e2] батч 400/2438 loss 1.6473 95s
  [bge5f_e2] батч 500/2438 loss 1.6174 116s
  [bge5f_e2] батч 600/2438 loss 1.5903 136s
  [bge5f_e2] батч 700/2438 loss 1.5855 157s
  [bge5f_e2] батч 800/2438 loss 1.5844 176s
  [bge5f_e2] батч 900/2438 loss 1.5700 198s
  [bge5f_e2] батч 1000/2438 loss 1.5607 217s
  [bge5f_e2] батч 1100/2438 loss 1.5520 238s
  [bge5f_e2] батч 1200/2438 loss 1.5574 257s
  [bge5f_e2] батч 1300/2438 loss 1.5351 278s
  [bge5f_e2] батч 1400/2438 loss 1.5139 297s
  [bge5f_e2] батч 1500/2438 loss 1.4947 318s
  [bge5f_e2] батч 1600/2438 loss 1.4756 337s
  [bge5f_e2] батч 1700/2438 loss 1.4622 358s
  [bge5f_e2] батч 1800/2438 loss 1.4430 377s
  [bge5f_e2] батч 1900/2438 loss 1.4271 398s
  [bge5f_e2] батч 2000/2438 loss 1.4194 418s
  [bge5f_e2] батч 2100/2438 loss 1.4040 439s
  [bge5f_e2] батч 2200/2438 loss 1.3916 458s
  [bge5f_e2] батч 2300/

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 710.04it/s]
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


  [bge5f_e2] батч 100/2472 loss 1.8298 20s
  [bge5f_e2] батч 200/2472 loss 1.6569 41s
  [bge5f_e2] батч 300/2472 loss 1.6327 63s
  [bge5f_e2] батч 400/2472 loss 1.6120 82s
  [bge5f_e2] батч 500/2472 loss 1.6015 103s
  [bge5f_e2] батч 600/2472 loss 1.5991 122s
  [bge5f_e2] батч 700/2472 loss 1.5917 143s
  [bge5f_e2] батч 800/2472 loss 1.5811 162s
  [bge5f_e2] батч 900/2472 loss 1.5800 183s
  [bge5f_e2] батч 1000/2472 loss 1.5785 202s
  [bge5f_e2] батч 1100/2472 loss 1.5734 223s
  [bge5f_e2] батч 1200/2472 loss 1.5664 243s
  [bge5f_e2] батч 1300/2472 loss 1.5452 263s
  [bge5f_e2] батч 1400/2472 loss 1.5216 283s
  [bge5f_e2] батч 1500/2472 loss 1.4969 304s
  [bge5f_e2] батч 1600/2472 loss 1.4750 322s
  [bge5f_e2] батч 1700/2472 loss 1.4592 343s
  [bge5f_e2] батч 1800/2472 loss 1.4434 362s
  [bge5f_e2] батч 1900/2472 loss 1.4285 384s
  [bge5f_e2] батч 2000/2472 loss 1.4151 403s
  [bge5f_e2] батч 2100/2472 loss 1.4029 424s
  [bge5f_e2] батч 2200/2472 loss 1.3929 444s
  [bge5f_e2] батч 2300/

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 718.98it/s]
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
C:\Users\Mikhail\AppData\Local\Temp\ipykernel_9688\81505692.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)


  [bge5f_e2] батч 100/2432 loss 1.9172 19s
  [bge5f_e2] батч 200/2432 loss 1.7599 37s
  [bge5f_e2] батч 300/2432 loss 1.6904 57s
  [bge5f_e2] батч 400/2432 loss 1.6389 75s
  [bge5f_e2] батч 500/2432 loss 1.6067 95s
  [bge5f_e2] батч 600/2432 loss 1.6105 113s
  [bge5f_e2] батч 700/2432 loss 1.5897 132s
  [bge5f_e2] батч 800/2432 loss 1.5708 150s
  [bge5f_e2] батч 900/2432 loss 1.5553 170s
  [bge5f_e2] батч 1000/2432 loss 1.5549 188s
  [bge5f_e2] батч 1100/2432 loss 1.5425 208s
  [bge5f_e2] батч 1200/2432 loss 1.5487 226s
  [bge5f_e2] батч 1300/2432 loss 1.5285 246s
  [bge5f_e2] батч 1400/2432 loss 1.5031 264s
  [bge5f_e2] батч 1500/2432 loss 1.4757 284s
  [bge5f_e2] батч 1600/2432 loss 1.4566 302s
  [bge5f_e2] батч 1700/2432 loss 1.4390 322s
  [bge5f_e2] батч 1800/2432 loss 1.4267 340s
  [bge5f_e2] батч 1900/2432 loss 1.4171 360s
  [bge5f_e2] батч 2000/2432 loss 1.4091 378s
  [bge5f_e2] батч 2100/2432 loss 1.3996 398s
  [bge5f_e2] батч 2200/2432 loss 1.3890 418s
  [bge5f_e2] батч 2300/2

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 711.47it/s]
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


  [bge5f_e2] батч 100/2434 loss 1.8273 21s
  [bge5f_e2] батч 200/2434 loss 1.7434 42s
  [bge5f_e2] батч 300/2434 loss 1.6689 63s
  [bge5f_e2] батч 400/2434 loss 1.6421 82s
  [bge5f_e2] батч 500/2434 loss 1.6260 102s
  [bge5f_e2] батч 600/2434 loss 1.6026 121s
  [bge5f_e2] батч 700/2434 loss 1.5952 142s
  [bge5f_e2] батч 800/2434 loss 1.5902 161s
  [bge5f_e2] батч 900/2434 loss 1.5801 182s
  [bge5f_e2] батч 1000/2434 loss 1.5830 202s
  [bge5f_e2] батч 1100/2434 loss 1.5724 223s
  [bge5f_e2] батч 1200/2434 loss 1.5645 242s
  [bge5f_e2] батч 1300/2434 loss 1.5376 263s
  [bge5f_e2] батч 1400/2434 loss 1.5143 282s
  [bge5f_e2] батч 1500/2434 loss 1.4956 303s
  [bge5f_e2] батч 1600/2434 loss 1.4789 322s
  [bge5f_e2] батч 1700/2434 loss 1.4634 343s
  [bge5f_e2] батч 1800/2434 loss 1.4496 362s
  [bge5f_e2] батч 1900/2434 loss 1.4353 382s
  [bge5f_e2] батч 2000/2434 loss 1.4211 402s
  [bge5f_e2] батч 2100/2434 loss 1.4096 423s
  [bge5f_e2] батч 2200/2434 loss 1.3951 442s
  [bge5f_e2] батч 2300/

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 709.09it/s]
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
C:\Users\Mikhail\AppData\Local\Temp\ipykernel_9688\81505692.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)


  [bge5f_e2] батч 100/2394 loss 2.0048 21s
  [bge5f_e2] батч 200/2394 loss 1.8410 42s
  [bge5f_e2] батч 300/2394 loss 1.7464 63s
  [bge5f_e2] батч 400/2394 loss 1.7161 82s
  [bge5f_e2] батч 500/2394 loss 1.6642 103s
  [bge5f_e2] батч 600/2394 loss 1.6484 122s
  [bge5f_e2] батч 700/2394 loss 1.6179 143s
  [bge5f_e2] батч 800/2394 loss 1.6168 162s
  [bge5f_e2] батч 900/2394 loss 1.6151 183s
  [bge5f_e2] батч 1000/2394 loss 1.5990 202s
  [bge5f_e2] батч 1100/2394 loss 1.5738 223s
  [bge5f_e2] батч 1200/2394 loss 1.5685 242s
  [bge5f_e2] батч 1300/2394 loss 1.5355 262s
  [bge5f_e2] батч 1400/2394 loss 1.5164 282s
  [bge5f_e2] батч 1500/2394 loss 1.4947 303s
  [bge5f_e2] батч 1600/2394 loss 1.4722 321s
  [bge5f_e2] батч 1700/2394 loss 1.4600 342s
  [bge5f_e2] батч 1800/2394 loss 1.4454 360s
  [bge5f_e2] батч 1900/2394 loss 1.4309 380s
  [bge5f_e2] батч 2000/2394 loss 1.4147 398s
  [bge5f_e2] батч 2100/2394 loss 1.4001 418s
  [bge5f_e2] батч 2200/2394 loss 1.3898 436s
  [bge5f_e2] батч 2300/

## Второй кросс-энкодер 

In [ ]:
ce_mini = run_ce('rubert5f','DiTy/cross-encoder-russian-msmarco',maxlen=384,epochs=3,lr=3e-5,qpb=4,accum=2,pred_bs=128,seed_off=1000,grad_ckpt=False)
gc.collect(); torch.cuda.empty_cache()
print('rubert OOF NDCG:',ndcg10(alldf.iloc[tr_idx],ce_mini[tr_idx]))
print('среднее z двух CE:',ndcg10(alldf.iloc[tr_idx],(pd.Series(ce_bge).groupby(alldf.query_id.values).transform(lambda x:(x-x.mean())/(x.std()+1e-6))+pd.Series(ce_mini).groupby(alldf.query_id.values).transform(lambda x:(x-x.mean())/(x.std()+1e-6))).values[tr_idx]))


c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mikhail\.cache\huggingface\hub\models--DiTy--cross-encoder-russian-msmarco. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


=== [rubert5f] фолд 1/5 ===


c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mikhail\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1260.43it/s]
C:\Users\Mikhail\AppData\Local\Temp\ipykernel_9688

  [rubert5f] батч 100/1830 loss 2.1135 9s
  [rubert5f] батч 200/1830 loss 2.0397 18s
  [rubert5f] батч 300/1830 loss 2.0134 28s
  [rubert5f] батч 400/1830 loss 1.9744 37s
  [rubert5f] батч 500/1830 loss 1.9381 46s
  [rubert5f] батч 600/1830 loss 1.9155 56s
  [rubert5f] батч 700/1830 loss 1.8497 65s
  [rubert5f] батч 800/1830 loss 1.7814 75s
  [rubert5f] батч 900/1830 loss 1.7426 84s
  [rubert5f] батч 1000/1830 loss 1.7098 93s
  [rubert5f] батч 1100/1830 loss 1.6850 103s
  [rubert5f] батч 1200/1830 loss 1.6654 112s
  [rubert5f] батч 1300/1830 loss 1.6366 122s
  [rubert5f] батч 1400/1830 loss 1.6010 131s
  [rubert5f] батч 1500/1830 loss 1.5730 141s
  [rubert5f] батч 1600/1830 loss 1.5510 150s
  [rubert5f] батч 1700/1830 loss 1.5257 159s
  [rubert5f] батч 1800/1830 loss 1.5036 168s
[rubert5f] fold NDCG@10: 0.8922204661728856
=== [rubert5f] фолд 2/5 ===


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1852.46it/s]
C:\Users\Mikhail\AppData\Local\Temp\ipykernel_9688\81505692.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)


  [rubert5f] батч 100/1854 loss 2.2311 9s
  [rubert5f] батч 200/1854 loss 2.1022 18s
  [rubert5f] батч 300/1854 loss 2.0523 28s
  [rubert5f] батч 400/1854 loss 2.0083 37s
  [rubert5f] батч 500/1854 loss 1.9536 47s
  [rubert5f] батч 600/1854 loss 1.9176 56s
  [rubert5f] батч 700/1854 loss 1.8587 66s
  [rubert5f] батч 800/1854 loss 1.8057 75s
  [rubert5f] батч 900/1854 loss 1.7628 85s
  [rubert5f] батч 1000/1854 loss 1.7211 94s
  [rubert5f] батч 1100/1854 loss 1.6961 105s
  [rubert5f] батч 1200/1854 loss 1.6738 116s
  [rubert5f] батч 1300/1854 loss 1.6400 128s
  [rubert5f] батч 1400/1854 loss 1.6048 140s
  [rubert5f] батч 1500/1854 loss 1.5782 151s
  [rubert5f] батч 1600/1854 loss 1.5522 161s
  [rubert5f] батч 1700/1854 loss 1.5249 172s
  [rubert5f] батч 1800/1854 loss 1.5059 182s
[rubert5f] fold NDCG@10: 0.8885106530594294
=== [rubert5f] фолд 3/5 ===


... (2 outputs left out)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1654.74it/s]


  [rubert5f] батч 100/1827 loss 2.1628 10s
  [rubert5f] батч 200/1827 loss 2.0585 20s
  [rubert5f] батч 300/1827 loss 1.9835 29s
  [rubert5f] батч 400/1827 loss 1.9545 39s
  [rubert5f] батч 500/1827 loss 1.9290 49s
  [rubert5f] батч 600/1827 loss 1.9133 59s
  [rubert5f] батч 700/1827 loss 1.8575 69s
  [rubert5f] батч 800/1827 loss 1.8000 78s
  [rubert5f] батч 900/1827 loss 1.7588 88s
  [rubert5f] батч 1000/1827 loss 1.7242 98s
  [rubert5f] батч 1100/1827 loss 1.6917 108s
  [rubert5f] батч 1200/1827 loss 1.6707 117s
  [rubert5f] батч 1300/1827 loss 1.6402 128s
  [rubert5f] батч 1400/1827 loss 1.6055 137s
  [rubert5f] батч 1500/1827 loss 1.5772 148s
  [rubert5f] батч 1600/1827 loss 1.5491 159s
  [rubert5f] батч 1700/1827 loss 1.5248 172s
  [rubert5f] батч 1800/1827 loss 1.5035 183s
[rubert5f] fold NDCG@10: 0.8948336070567817
=== [rubert5f] фолд 5/5 ===


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1417.79it/s]


  [rubert5f] батч 100/1797 loss 2.1232 10s
  [rubert5f] батч 200/1797 loss 2.0366 20s
  [rubert5f] батч 300/1797 loss 1.9818 32s
  [rubert5f] батч 400/1797 loss 1.9562 43s
  [rubert5f] батч 500/1797 loss 1.9304 53s
  [rubert5f] батч 600/1797 loss 1.9231 63s
  [rubert5f] батч 700/1797 loss 1.8624 73s
  [rubert5f] батч 800/1797 loss 1.8064 83s
  [rubert5f] батч 900/1797 loss 1.7688 93s
  [rubert5f] батч 1000/1797 loss 1.7278 102s
  [rubert5f] батч 1100/1797 loss 1.7012 113s
  [rubert5f] батч 1200/1797 loss 1.6799 122s
  [rubert5f] батч 1300/1797 loss 1.6438 132s
  [rubert5f] батч 1400/1797 loss 1.6088 142s
  [rubert5f] батч 1500/1797 loss 1.5773 152s
  [rubert5f] батч 1600/1797 loss 1.5578 162s
  [rubert5f] батч 1700/1797 loss 1.5336 172s
[rubert5f] fold NDCG@10: 0.8954674880784405
rubert OOF NDCG: 0.8906278932147627
среднее z двух CE: 0.9272310601423536


## Признаки CE и статьи

In [ ]:
F['ce']=ce_bge          
F['ce_mini']=ce_mini
HAVE_MINI=True
g=alldf.query_id; qid,aid=alldf.query_id,alldf.article_id
def zs(c): return (F[c]-F.groupby(g)[c].transform('mean'))/(F.groupby(g)[c].transform('std').fillna(0)+1e-6)
F['ce_rank']=F.groupby(g).ce.rank(ascending=False,method='min'); F['ce_z']=zs('ce'); F['ce_dmax']=F.ce-F.groupby(g).ce.transform('max')
base=['bm_text','tf_c_text','ov_text_r','tf_c_title','ce']
if HAVE_MINI:
    F['ce_mini_rank']=F.groupby(g).ce_mini.rank(ascending=False,method='min'); F['ce_mini_z']=zs('ce_mini')
    F['ce_mini_dmax']=F.ce_mini-F.groupby(g).ce_mini.transform('max'); F['ce_avg_z']=F.ce_z+F.ce_mini_z
    base+=['ce_mini','ce_avg_z']
F['pi_missing']=alldf.passage_index.isna().astype(float)
for c in base:
    ga=F.groupby([qid,aid])[c]
    F[c+'_a_max']=ga.transform('max'); F[c+'_a_mean']=ga.transform('mean'); F[c+'_a_gap']=F[c]-F[c+'_a_max']
    F[c+'_a_rank']=F.groupby(qid)[c+'_a_max'].rank(ascending=False,method='dense')
    t=pd.DataFrame({'q':qid,'a':aid,'pi':alldf.passage_index,'v':F[c]}).sort_values(['q','a','pi'])
    gg=t.groupby(['q','a']).v; F[c+'_prev']=gg.shift(1); F[c+'_next']=gg.shift(-1)
print(F.shape)

(45037, 95)


## CatBoost

In [ ]:

Xtr=F.iloc[tr_idx].reset_index(drop=True); Xte=F.iloc[te_idx].reset_index(drop=True)
dtr=alldf.iloc[tr_idx].reset_index(drop=True); dte=alldf.iloc[te_idx].reset_index(drop=True)
ytr=dtr.relevance.astype(int).values; qtr=dtr.query_id.values
cb_params=dict(loss_function='YetiRank',eval_metric='NDCG:top=10',iterations=2000,learning_rate=0.05,depth=6,
               l2_leaf_reg=3,early_stopping_rounds=100,random_seed=SEED,thread_count=-1,verbose=0)
def run_cv(cols, Xa, Xb):
    oof=np.zeros(len(Xa)); pt=np.zeros(len(Xb)); its=[]
    for tri,vai in folds:
        tri=tri[np.argsort(qtr[tri],kind='stable')]; vai=vai[np.argsort(qtr[vai],kind='stable')]
        m=CatBoostRanker(**cb_params)
        m.fit(Pool(Xa.loc[tri,cols],ytr[tri],group_id=qtr[tri]),eval_set=Pool(Xa.loc[vai,cols],ytr[vai],group_id=qtr[vai]),use_best_model=True)
        oof[vai]=m.predict(Xa.loc[vai,cols]); its.append(m.get_best_iteration()); pt+=m.predict(Xb[cols])/len(folds)
    return oof,pt,its
print('bge CE only :',ndcg10(dtr,Xtr.ce.values))
cols1=list(Xtr.columns)
oofs,pts=[],[]
for sd in [42,7,123]:
    cb_params['random_seed']=sd
    o,p,its=run_cv(cols1,Xtr,Xte); oofs.append(o); pts.append(p); print('seed',sd,ndcg10(dtr,o),its)
oof1=np.mean(oofs,0); pt1=np.mean(pts,0)
print('Stage1 (3 seeds):',ndcg10(dtr,oof1))

def run_lgb(cols):
    oof=np.zeros(len(Xtr)); pt=np.zeros(len(Xte))
    for tri,vai in folds:
        tri=tri[np.argsort(qtr[tri],kind='stable')]; vai=vai[np.argsort(qtr[vai],kind='stable')]
        gt=pd.Series(qtr[tri]).groupby(qtr[tri],sort=False).size().values; gv=pd.Series(qtr[vai]).groupby(qtr[vai],sort=False).size().values
        m=lgb.LGBMRanker(objective='lambdarank',n_estimators=1500,learning_rate=0.03,num_leaves=15,min_child_samples=30,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=5,random_state=SEED,verbose=-1)
        m.fit(Xtr.loc[tri,cols],ytr[tri],group=gt,eval_set=[(Xtr.loc[vai,cols],ytr[vai])],eval_group=[gv],eval_at=[10],callbacks=[lgb.early_stopping(100,verbose=False)])
        oof[vai]=m.predict(Xtr.loc[vai,cols]); pt+=m.predict(Xte[cols])/len(folds)
    return oof,pt
oof_l,pt_l=run_lgb(cols1); print('LightGBM:',ndcg10(dtr,oof_l))
def qz(v,q): return pd.Series(v).groupby(q).transform(lambda x:(x-x.mean())/(x.std()+1e-6)).values
qte=dte.query_id.values
best=(0,1.0)
for w in [0,0.2,0.3,0.4,0.5,0.6,0.7,0.8,1.0]:
    sc=ndcg10(dtr,w*qz(oof1,qtr)+(1-w)*qz(oof_l,qtr)); print('w_cat',w,round(sc,5))
    if sc>best[0]: best=(sc,w)
w=best[1]; print('лучший вес CatBoost:',w)
oof_final=w*qz(oof1,qtr)+(1-w)*qz(oof_l,qtr); pt1=w*qz(pt1,qte)+(1-w)*qz(pt_l,qte)
print('Финал OOF (бленд):',ndcg10(dtr,oof_final)); oof1=oof_final


bge CE only : 0.9345777791790929
seed 42 0.9405665839263331 [208, 26, 44, 338, 83]
seed 7 0.9405688172807508 [322, 130, 47, 317, 138]
seed 123 0.9405647385450072 [160, 148, 273, 320, 99]
Stage1 (3 seeds): 0.9399181954580113


c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Mikhail\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is

LightGBM: 0.9416318392652985
w_cat 0 0.94163
w_cat 0.2 0.94056
w_cat 0.3 0.94055
w_cat 0.4 0.94065
w_cat 0.5 0.94027
w_cat 0.6 0.9406
w_cat 0.7 0.93998
w_cat 0.8 0.93989
w_cat 1.0 0.93992
лучший вес CatBoost: 0
Финал OOF (бленд): 0.9416318392652985


## Submission

In [11]:
d = pd.DataFrame({'id': dte.id.values, 'prediction': pt1})
out = sub[['id']].merge(d, on='id', how='left'); assert out.prediction.notna().all() and len(out) == len(sub)
out.to_csv('submission.csv', index=False); np.save('pt_5fold.npy', pt1); np.save('oof_5fold.npy', oof1)
print('CV CatBoost (5 фолдов):', ndcg10(dtr, oof1)); print('сохранён submission.csv', out.shape)

CV CatBoost (5 фолдов): 0.9416318392652985
сохранён submission.csv (13507, 2)
